<font size="+4">`Signal and Audio Processing`</font>

<font size="+3">`Seminar 09: Intro to Generative Text-to-Speech`</font>

<font size="+2">`Maks Nakhodnov & Dmitry Kropotov`</font>

<font size="+2">`Bremen, 2026`</font>

Что вы узнаете из этого ноутбука:

* **Постановка задачи TTS:** Проблема *One-to-Many*. Оценка генерации речи.
* **Современные подходы к TTS:** от классических mel-спектрограмм и AR/NAR моделей до дискретных аудио-токенов и LLM-based архитектур.

In [1]:
import io
import os

os.environ['TORCH_ROCM_AOTRITON_ENABLE_EXPERIMENTAL'] = '1'

from urllib.request import urlopen

import regex

import numpy as np

import torch
import torchaudio
import torchmetrics
import torch.nn.functional as F
from speechbrain.pretrained import EncoderClassifier
from transformers import SpeechT5Processor, SpeechT5ForTextToSpeech, SpeechT5HifiGan

import soundfile

import matplotlib
import matplotlib.pyplot as plt

from IPython.display import Audio

import matplotlib_inline
matplotlib_inline.backend_inline.set_matplotlib_formats('svg')

/tmp/ipykernel_34121/26956645.py:16: UserWarning: Module 'speechbrain.pretrained' was deprecated, redirecting to 'speechbrain.inference'. Please update your script. This is a change from SpeechBrain 1.0. See: https://github.com/speechbrain/speechbrain/releases/tag/v1.0.0
  from speechbrain.pretrained import EncoderClassifier


# `1. Постановка задачи TTS и Проблема One-to-Many`

**Text-to-Speech (TTS)** — задача генерации человеческой речи из текста. Если ASR — это задача *распознавания* (сжатия непрерывного зашумленного сигнала в дискретные смысловые токены), то TTS — это задача **генерации**, требующая восстановления утерянной информации.

Пусть $Y = (y_1, y_2, \dots, y_N)$ — входная текстовая последовательность, где $y_n \in V$ (буквы, фонемы или токены). 
Задача TTS — сгенерировать аудиосигнал $S = (\mathbf{s}_1, \dots, \mathbf{s}_T)$ или последовательность акустических признаков $X = (\mathbf{x}_1, \dots, \mathbf{x}_M)$, максимизируя вероятность:

$$ X^* = \arg\max_{X} P(X \mid Y, \mathcal{C}) $$

Где $\mathcal{C}$ — контекстное условие (личность спикера, эмоция, скорость, акустика помещения).

## `1.1. The One-to-Many Problem`

Главная фундаментальная сложность TTS заключается в том, что отображение текста в звук **не инъективно**. Одну и ту же фразу «Привет, как дела?» можно произнести бесконечным числом способов:
* **Pitch, Timbre:** С разным тембром (мужчина, женщина, ребенок).
* **Prosody, Speed:** С разной скоростью и паузами.
* **Energy, Emotions:** С разной интонацией (сарказм, радость, грусть).
* **Environment:** В разных окружениях, с разным фоновым шумом.

<b style='color:red;'>Почему в TTS нельзя просто считать L1/MSE Loss между предсказанной и эталонной спектрограммой, как это делается, например, при генерации картинок?</b>

<details><summary>Ответ:</summary>>> 
    
1. Проблема выравнивания во времени: Речь, сгенерированная чуть быстрее или медленнее эталона, даст большой MSE-лосс, даже если звучит идеально для человека. <br>
2. MSE штрафует модель за выбор другой, но валидной интонации. Модель обучается предсказывать математическое ожидание всех спектрограмм для данного текста, что физически выглядит как размытая спектрограмма без четких гармоник. Такой эффект *Oversmoothing* приводит к генерации скучного, роботизированного голоса.
</details>

## `1.2. Структура TTS пайплайна`

<img src="ASR_vs_TTS.svg" alt="" >

Существует множество вариантов, как выбрать каждую из компонент в TTS пайплайне:

1. **Pre-processing:**
    * Текст $\to$ Нормализация $\to$ Фонемы $\to$ Извлечение просодии.
    * Текст $\to$ BPE-токены \& Аудио-референс $\to$ Дискретные аудио-токены.
    * ...
2. **Acoustic Model:**
    * Фонемы $\to$ Mel-спектрограмма (модели *Tacotron 2*, *FastSpeech 2*).
    * Совместное моделирование семантики и акустики (LLM или Flow-модели).
    * ...
3. **Acoustic Features:**
    * Mel/Linear-спектрограмма
    * Латентные скрытые представления
    * Discrete audio codes
    * ...
5. **Vocoder:**
    * Mel-спектрограмма $\to$ Waveform (модели *HiFi-GAN*, *Vocos*).
    * Декодирование токенов или латентов обратно в сырой звук.
    * ...

Каким бы ни было промежуточное представление (спектрограмма или токены), финальным шагом всегда является конвертация в сырую звуковую волну — одномерный массив амплитуд с частотой $24\,000 - 48\,000$ отсчетов в секунду. Этим занимается **Vocoder**.

<b style='color:red;'>Почему нельзя отказаться от вокодера и заставить акустическую модель предсказывать Raw Waveform напрямую (например, авторегрессионно)?</b>

<details><summary>Ответ:</summary>>> 
Из-за частоты дискретизации. Для 1 секунды аудио с частотой 24 кГц модели нужно сделать 24000 авторегрессионных шагов. WaveNet делала именно так, но её инференс занимал минуты для секундного фрагмента. Вокодеры решают проблему "разрешения": акустическая модель работает на низкой частоте (\~50 Гц), предсказывая смысл и мелодию, а вокодер параллельно генерирует высокочастотные акустические детали до 24 кГц.
</details>

## `1.3. Text Frontend: Нормализация и G2P`

Прежде чем акустическая модель начнет генерировать звук, сырой текст необходимо подготовить. Несмотря на то, что современные LLM-based подходы могут использовать информацию о структуре языка для E2E препроцессинга текста, качественная предобработка всё ещё важна для реальных систем.

### `Text Normalization`

Модели TTS не умеют читать цифры, символы и аббревиатуры напрямую. Нормализация переводит небуквенные символы в их словесное (орфографическое) представление:

| Сырой текст | Контекст | Нормализованный текст (как нужно прочитать) |
| :--- | :--- | :--- |
| `1984` | В **1984** году вышел роман... | *в тысяча девятьсот восемьдесят четвертом* |
| `1984` | Мой пин-код: **1984** | *один девять восемь четыре* |
| `St.` | **St.** Patrick's Day | *Saint* |
| `St.` | Wall **St.** | *Street* |
| `$120.5` | Цена: **$120.5** | *сто двадцать долларов пятьдесят центов* |

**Важно отметить, что эта задача нетривиальна, так как результат разрешения неоднозначностей зависит от контекста!.**

Исторически для решения этой задачи использовались своды правил, написанные лингвистами вручную. Сегодня применяются нейронные модели или LLM, работающие в режиме перевода сырого текста в нормализованный.

<b style='color:red;'>В чём опасность использования нейросетей для текстовой нормализации?</b>

<details><summary>Ответ:</summary>>> 
    
<b>Галлюцинации и пропуск слов.</b> Если ручные правила не знают слово, оно просто будет прочитано по буквам. LLM же может перефразировать текст, выбросить "неудобную" цифру или заменить её на случайную. Поэтому в индустрии до сих пор преобладают гибридные Rule-based + Neural подходы.

</details>

### `Grapheme-to-Phoneme (G2P)`

Буквы (графемы) не всегда однозначно отражают звучание слова. В английском языке одно и то же буквосочетание может читаться десятками способов (e.g., *read* $\to$ /riːd/ или /rɛd/). В русском языке есть редукция гласных (корова $\to$[к а р о́ в а]) и **омографы** (слова, которые пишутся одинаково, но звучат по-разному из-за ударения):

* Я открыл дверной **замóк**.
* Мы поехали на экскурсию в старый **зáмок**.

G2P-модели расставляют ударения и переводят нормализованный текст в транскрипцию.

<b style='color:red;'>Зачем вообще переводить текст в фонемы, если современные Audio LLM могут напрямую работать с BPE-токенами текста?</b>

<details><summary>Ответ:</summary>>> 
    
Использование BPE действительно упрощает пайплайн, и модели учатся неявно связывать токены со звуками. Однако:
1. BPE разбивает слова логически, а не фонетически. Редкие слова, имена собственные или новые заимствования модель прочитает с ошибкой ударения или произношения.
2. Фонемы делают модель более стабильной и позволяют вручную исправлять ошибки произношения в словарях без переобучения всей акустической модели. Тем не менее, SOTA модели все больше склоняются к отказу от явного G2P при масштабировании данных.

</details>

Больше дискуссии по этому вопросу [здесь](https://huggingface.co/blog/hexgrad/g2p).

## `1.4. Методы оценивания и метрики`

### `Метрики производительности`

Оценка скорости TTS пайплайна следует аналогичному пути, что и для ASR:

1. **RTF (Real-Time Factor):**
  
   Это отношение времени, затраченного на обработку аудио, к длительности самого аудио:

   $$\text{RTF} = \frac{\text{Processing Time}}{\text{Audio Duration}}$$
    У современных Flow Matching моделей $\text{RTF} \approx 0.05$.

    Аналогично ASR иногда рассматривают обратную величину: $\text{RTFx} = \frac{1}{\text{RTF}}$

2. **First Chunk Latency:**

   Также как для Streaming ASR было важно время до первого токена, для TTS измеряют время до первого сгенерированного чанка.

<img src="https://pic3.zhimg.com/v2-29920fb92ba21fdc5fcdb6270e1e141c_1440w.jpg" alt="" >

### `Объективные метрики качества`

В задаче генерации нет единой математической метрики качества, так как «натуральность» субъективна. Исторически использовались акустические метрики, но сегодня индустрия перешла к гибридным и LLM-based подходам.

1. **Word Error Rate / CER:**

   Измеряет *Разборчивость* и стабильность. Сгенерированное аудио распознаётся через ASR пайплайн с помощью сильных моделей, обычно *Whisper*. Если генеративная модель "проглатывает" буквы, зацикливается или галлюцинирует, то WER вырастет.

<b style='color:red;'>Почему такой подход будет работать? Разве ASR не добавит своих ошибок распознавания?</b>

<details><summary>Ответ:</summary>>> 

Обычно TTS генерация — генерация аудио студийного качества. Для домена высококачественных аудио без постороннего шума ошибка современных ASR моделей минимальна.
 
</details>
   
2. **Speaker Similarity (SS):**

   Измеряет качество *Zero-Shot Voice Cloning*. Эмбеддинги спикера из оригинального референсного аудио и из сгенерированного аудио извлекаются с помощью предобученной модели. 
   $$ \text{SIM} = \cos(\mathbf{e}_{\text{ref}}, \mathbf{e}_{\text{gen}}) = \frac{\mathbf{e}_{\text{ref}} \cdot \mathbf{e}_{\text{gen}}}{\|\mathbf{e}_{\text{ref}}\| \|\mathbf{e}_{\text{gen}}\|} $$
   Значения $\text{SIM} > 0.7$ обычно означают, что человек не отличит голоса.

| Model | Text token | Speech Token | WER (%) | #INS+DEL | #SUB | SS |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| Original | - | - | 3.01 | 66 | 200 | 69.67 |
| VALL-E (Wang et al., 2023) | Phone | Encodec | 18.70 | 342 | 1312 | 53.19 |
| UniAudio (Yang et al., 2023) | Phone | Encodec | 8.74 | 254 | 519 | 47.56 |
| SpearTTS (Kharitonov et al., 2023) | Phone | Hubert | 6.14 | 133 | 410 | 51.71 |
| Exp-1-LibriTTS | Phone | Hubert | 7.41 | 325 | 409 | 67.85 |
| Exp-2-LibriTTS | Phone | S<sup>3</sup><sub>en</sub> | 5.05 | 122 | 325 | 67.85 |
| Exp-3-LibriTTS | BPE<sub>en</sub> | S<sup>3</sup><sub>en</sub> | 3.93 | 108 | 239 | 67.85 |
| Exp-4-LibriTTS | BPE | S<sup>3</sup> | 4.76 | 134 | 287 | 65.94 |
| Exp-4-Large-scale | BPE | S<sup>3</sup> | **3.17** | **96** | **184** | **69.49** |

In [2]:
from transformers import Wav2Vec2FeatureExtractor, WavLMForXVector

@torch.no_grad()
def get_embedding(audio_path, feature_extractor, model):
    audio, sample_rate = soundfile.read(audio_path)
    audio = torch.tensor(audio)

    audio = torchaudio.functional.resample(
        audio, orig_freq=sample_rate, new_freq=16000
    )
    inputs = feature_extractor(
        audio.numpy(), return_tensors="pt", sampling_rate=16000
    )
    
    [embedding] = model(**inputs).embeddings
    return embedding / torch.linalg.norm(embedding)

In [3]:
model_name = "microsoft/wavlm-base-plus-sv"
feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(model_name)
model = WavLMForXVector.from_pretrained(model_name).eval()

emb1 = get_embedding('./audio_test.wav', feature_extractor, model)
emb2 = get_embedding('./../../Tasks/Task 01/data/singing.wav', feature_extractor, model)
torch.dot(emb1, emb2).item()

/home/maksim64/miniconda3/envs/torch/lib/python3.12/site-packages/torch/nn/functional.py:6044: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(


0.2793833017349243

3. [**Fréchet DeepSpeech Distance (FDSD) / Kernel DeepSpeech Distance (KDSD):**](https://arxiv.org/pdf/1909.11646)

   Аналог FID/KID в CV. Измеряет дистанцию между распределениями реальной речи и сгенерированной в пространстве признаков предобученной ASR.
   $$ \text{FDSD} = \mathcal{W}_{2}(\mathcal{P}_{\text{reference}}, \mathcal{P}_{\text{source}}) \approx \mathcal{W}_{2}(\mathcal{N}(\mu_r, \Sigma_r), \mathcal{N}(\mu_s, \Sigma_s))$$

   $$
    \text{KDSD} = \text{MMD}(f_{\text{reference}}, f_{\text{source}})^{2}
   $$

### `Субъективные метрики качества`

1. **MOS (Mean Opinion Score):** Оценка людьми от 1 до 5 (обычно, по параметрам Naturalness и Speaker Similarity). Долго и дорого.

| Model | MOS | FDSD | cFDSD | KDSD ×10<sup>5</sup> | cKDSD ×10<sup>5</sup> |
| :--- | :--- | :--- | :--- | :--- | :--- |
| *natural speech* | 4.55 ± 0.075 | 0.161 | N/A | 0 | 0 |
| *WaveNet*, van den Oord et al. (2016) | 4.41 ± 0.069 | | | | |
| *Parallel WaveNet*, van den Oord et al. (2018) | 4.41 ± 0.078 | | | | |
| FullD | 1.889 ± 0.057 | 4.51 | 4.46 | 785 | 782 |
| cRWD<sub>1</sub> | 3.394 ± 0.058 | 0.362 | 0.247 | 35.2 | 30.9 |
| cRWD<sub>{1,2,4,8,15}</sub> | 3.498 ± 0.059 | 0.398 | 0.284 | 42.1 | 37.9 |
| cRWD<sub>1</sub> + uRWD<sub>1</sub> | 3.502 ± 0.057 | 0.259 | 0.144 | 16.6 | 12.3 |
| (cRWD<sub>1</sub> + uRWD<sub>1</sub>)<sup>×5</sup> | 3.526 ± 0.054 | 0.194 | 0.073 | 5.59 | 1.34 |
| RWD<sub>1,240×{1,2,4,8,15}</sub> | 4.154 ± 0.050 | 0.184 | 0.061 | 3.73 | 0.54 |
| RWD<sup>*</sup><sub>480</sub> | 4.195 ± 0.045 | 0.193 | 0.069 | 5.28 | 0.98 |
| GAN-TTS (RWD<sup>*</sup>) | 4.213 ± 0.046 | 0.184 | 0.060 | 3.84 | 0.37 |

2. **ABX Test:** Слушателю дают аудио A (модель 1), аудио B (модель 2) и референс X. Нужно выбрать, что ближе к X.
3. [**MLLM-as-a-Judge**:](https://arxiv.org/pdf/2510.14664v1)

   С ростом популярности *Instruction-guided TTS* (когда модель использует сложные инструкции, например: «прочитай это саркастично»), классические метрики перестали работать. Сейчас стандартом является использование мультимодальных LLM в качестве автоматических асессоров.
    В MLLM загружают сгенерированное аудио, текст и промпт-инструкцию, после чего она оценивает:
    * **Instruction Following:** Выполнила ли модель сложную инструкцию (паузы, акцент, вздохи)?
    * **Expressiveness:** Эмоциональное богатство речи.

### `Датасеты и SOTA Бенчмарки`

В ASR для качественного и робастного распознавания аудио данные должны быть разнообразны. Однако для TTS картина обратная — если в аудио генерации будет произнесено не то, что подаётся на вход модели, TTS выучит неправильные произношения. Более того, unconditional модели без дополнительной информации о говорящем будут усреднять речь разных спикеров, что будет приводить к неестественной генерации. Поэтому для старых TTS моделей почти всегда использовались **чистые single speaker студийные датасеты**. Однако, современные модели, допускающие **условие по говорящему** (speaker embedding, style-токены, семантические подсказки и т.п.), эффективно обучаются на **многоголосых** наборах. Такие модели обучают на сотнях тысяч часов речи (из подкастов, YouTube), чтобы модель видела всё многообразие тембров и акустических условий.

<b style='color:red;'>Датасеты для ASR почти всегда используют частоту дискретизации 16 kHz. Почему для современных TTS датасетов стандартом является 24 kHz или даже 48 kHz?</b>

<details><summary>Ответ:</summary>>> 

1. В ASR главная цель — извлечь смысл. Вся основная лингвистическая и семантическая информация человеческой речи (форманты гласных и большинства согласных) лежит в диапазоне частот до 8 кГц. Согласно теореме Котельникова, частоты дискретизации 16 кГц достаточно, чтобы без потерь восстановить частоты до 8 кГц. 
2. В TTS цель — естественность и качество звучания. Частоты выше 8 кГц содержат высшие обертоны, дыхание, четкие шипящие звуки и индивидуальный окрас тембра. Если сгенерировать голос на 16 кГц, он будет звучать глухо. Для студийного чистого звучания требуется генерация высоких частот, поэтому TTS-датасеты и вокодеры работают на 24 кГц (покрывает частоты до 12 кГц) и выше.

</details>

<table style="width: 100%; border-collapse: collapse; font-family: sans-serif; text-align: left;">
  <tr style="background-color: #f2f2f2; font-weight: bold; border-bottom: 2px solid black;">
    <th style="padding: 10px;">Название датасета</th>
    <th style="padding: 10px;">Размер (Часы)</th>
    <th style="padding: 10px;">Домен / Особенности</th>
    <th style="padding: 10px;">Год</th>
  </tr>
  <tr style="border-bottom: 1px solid #ddd;">
    <td style="padding: 10px;"><b>LJSpeech</b></td>
    <td style="padding: 10px;">24</td>
    <td style="padding: 10px;">Студийная запись, 1 спикер. Исторический baseline для Tacotron/FastSpeech.</td>
    <td style="padding: 10px;">2017</td>
  </tr>
  <tr style="border-bottom: 1px solid #ddd;">
    <td style="padding: 10px;"><b>LibriTTS</b></td>
    <td style="padding: 10px;">585</td>
    <td style="padding: 10px;">Аудиокниги, 2456 спикеров. Стандарт для ранних multi-speaker моделей.</td>
    <td style="padding: 10px;">2019</td>
  </tr>
  <tr style="border-bottom: 1px solid #ddd;">
    <td style="padding: 10px;"><b>GigaSpeech</b></td>
    <td style="padding: 10px;">10,000</td>
    <td style="padding: 10px;">Подкасты, YouTube. Спонтанная речь, шумы. Дал старт zero-shot моделям.</td>
    <td style="padding: 10px;">2021</td>
  </tr>
  <tr style="border-bottom: 1px solid #ddd;">
    <td style="padding: 10px;"><b>Emilia</b></td>
    <td style="padding: 10px;">> 200,000</td>
    <td style="padding: 10px;">Масштабные In-the-wild датасеты на разных языках. Высокая вариативность.</td>
    <td style="padding: 10px;">2025</td>
  </tr>
  <tr style="border-bottom: 1px solid #ddd">
    <td style="padding: 10px;"><b>SpeechCraft / Parler-TTS</b></td>
    <td style="padding: 10px;">2,000 / 50,000</td>
    <td style="padding: 10px;"><b>Description-based:</b> Аудио сопровождается текстовым описанием ("A man speaks calmly...").</td>
    <td style="padding: 10px;">2024</td>
  </tr>
</table>

### `Примеры метрик для современных моделей`

Оценка моделей в генерации и в способности модели скопировать голос по 3-секундному референсу на бенчмарке LibriSpeech test-clean. Больше метрик можно [посмотреть здесь](https://huggingface.co/FunAudioLLM/Fun-CosyVoice3-0.5B-2512#evaluation).

| Модель | Архитектура | WER $\downarrow$<br>*(Один диктор / Zero-Shot)* | SS (Сходство голоса) $\uparrow$ | RTF $\downarrow$ |
| :--- | :--- | :---: | :---: | :---: |
| *Ground Truth* | *Human (LibriSpeech)* | *1.8 - 2.1* | *0.730* | *-* |
| [Tacotron 1](https://arxiv.org/abs/1703.10135) (2017) | RNN (Seq2Seq) + Griffin-Lim | \~4.0 / -  | низкое | \~0.50 - 1.0 |
| [Tacotron 2](https://arxiv.org/abs/1712.05884) (2017) | CNN + BiLSTM + Attention | \~2.5 / **>10.0** | низкое | > 1.0 (WaveNet)<br>\~0.10 (WaveGlow) |
| [FastSpeech 1](https://arxiv.org/abs/1905.09263) (2019) | Transformer (NAR) | \~3.0 / **>8.0** | низкое | ~0.03 |
|[FastSpeech 2](https://arxiv.org/abs/2006.04558) (2020) | Transformer (NAR) + Variance Adaptors| \~2.5 / **\~7.7** | низкое | **\~0.01 - 0.02** |
| [VALL-E](https://arxiv.org/abs/2301.02111) (2023) | AR + NAR | 5.9 | 0.580 | 0.50 |
| [VoiceBox](https://arxiv.org/abs/2306.15687) (2023) | Flow Matching | 1.9 | 0.681 | 0.64 |
| [CosyVoice](https://arxiv.org/abs/2407.05407) (2024)| LLM + Flow Matching | 3.2 | 0.695 | 0.92 |
| [F5-TTS](https://arxiv.org/abs/2410.06885) (2024) | Flow Matching | 2.0 | 0.647 | 0.15 |
|[Seed-TTS](https://arxiv.org/abs/2406.02430) (2024) | AR + DiT | 2.2 | **0.762** | 0.13 |

# `2. Представление аудио. Токенизация звука.`

Долгое время стандартом де-факто для работы со звуком в нейросетях была **мел-спектрограмма**. Хотя данное представление сохраняет почти всю физическую информацию о звуке, оно является **непрерывным**, что создает ряд критических проблем при попытке обучить по-настоящему генеративные модели:

1.  **Регрессия vs Классификация:** Работа со спектрограммой навязывает модели задачу регрессии — попытку предсказать точные значения амплитуд. Однако речь вариативна: одну и ту же фразу можно произнести с разной интонацией. Пытаясь минимизировать ошибку между всеми возможными вариантами, модель неизбежно стремится к среднему. Это приводит к **spectral blurring**, из-за чего синтезированный голос звучит роботизированно.
2.  **Аккумуляция ошибок и нестабильность:** В авторегрессионных моделях предсказание каждого следующего кадра спектрограммы зависит от предыдущих. В непрерывном пространстве ошибка на предыдущем кадре накапливается, и спектрограмма деградирует в физически невозможный сигнал. Дискретные представления обладают свойством самокоррекции: на каждом шаге модель обязана выбрать конкретный токен из словаря.
3.  **Отсутствие семантических абстракций:** Спектрограмма описывает физику процесса, но не его смысл. MSE на спектрограммах плохо коррелирует с человеческим восприятием: физически малая разница может звучать для человека как сильное искажение тембра. Нам же требуется представление, которое работало бы на уровне концептов (фонем, особенностей голоса и акустики) абстрагируясь от сырого звука.

Решением этих проблем стало использование **нейронных аудиокодеков**. Они позволяют перевести задачу из регрессии в классификацию. Такой подход превращает генерацию звука в задачу моделирования аудио-языка, где модель оперирует конечным словарем акустических единиц, а всю работу по восстановлению физической волны берет на себя предварительно обученный декодер.

## `2.1. Кванитизация непрерывных представлений`

Если попытаться создать словарь всех возможных звуков, его размер превысит миллиарды токенов. Чтобы обойти это ограничение, нейрокодеки используют **векторное квантование (VQ)**. 

Для этого непрерывный вектор $\mathbf{h}$ приближается вектором из словаря: $\mathbf{h} \approx \mathbf{c} \in C$. Словарь векторов (Codebook) может быть как обучаемый так и полученным за счёт детерминистичной процедуры.

Конкретный способ квантизации зависит от того, какой баланс между **Акустической** и **Семантической** информацией должна соблюдать модель:
* **Акустические токены** ориентированы на физические свойства аудиосигнала. Эти токены предназначены для кодирования низкоуровневых акустических характеристик, таких как высота тона, громкость, тембр, и так далее.
* **Семантические токены** выявляют смысловые единицы более высокого уровня. Эти токены определяют лингвистическую или контекстную информацию, а не только акустические свойства.

<div class="pst-scrollable-table-container"><table class="table">
<thead>
<tr class="row-odd"><th class="head"><p><strong>Аспект</strong></p></th>
<th class="head"><p><strong>Акустические токены</strong></p></th>
<th class="head"><p><strong>Семантические токены</strong></p></th>
</tr>
</thead>
<tbody>
<tr class="row-even"><td><p><strong>Фокус</strong></p></td>
<td><p>Низкоуровневые акустические свойства</p></td>
<td><p>Высокоуровневая семантическая информация</p></td>
</tr>
<tr class="row-odd"><td><p><strong>Цель</strong></p></td>
<td><p>Сжатие и реконструкция</p></td>
<td><p>Захват семантики</p></td>
</tr>
<tr class="row-even"><td><p><strong>Подход к обучению</strong></p></td>
<td><p>Unsupervised / Self-supervised</p></td>
<td><p>Self-supervised</p></td>
</tr>
<tr class="row-even"><td><p><strong>Прикладные задачи</strong></p></td>
<td><p>Лучше подходят для генерации и восстановления звука</p></td>
<td><p>Лучше подходят для задач классификации и понимания речи</p></td>
</tr>
<tr class="row-even"><td><p><strong>Эффективность сжатия</strong></p></td>
<td><p>Обычно выше</p></td>
<td><p>Может быть ниже, так как приоритет отдается семантике</p></td>
</tr>
<tr class="row-odd"><td><p><strong>Устойчивость к шуму</strong></p></td>
<td><p>Часто более чувствительны к акустическим вариациям</p></td>
<td><p>Могут быть более устойчивы к незначительным изменениям звука</p></td>
</tr>
<tr class="row-even"><td><p><strong>Многоязычность</strong></p></td>
<td><p>Обычно не зависят от языка</p></td>
<td><p>Могут фиксировать специфические особенности конкретного языка</p></td>
</tr>
</tbody>
</table>
</div>

<table style="width: 100%; border-collapse: collapse; font-family: sans-serif; table-layout: fixed;">
    <thead>
        <tr style="text-align: center; background-color: #f2f2f2;">
            <th style="padding: 15px; border: 1px solid #444;">
                <a href="https://arxiv.org/pdf/2210.13438">EnCodec: High Fidelity Audio Compression</a>
            </th>
            <th style="padding: 15px; border: 1px solid #444;">
                <a href="https://arxiv.org/pdf/2209.03143">AudioLM: a Language Modeling Approach to Audio Generation</a>
            </th>
            <th style="padding: 15px; border: 1px solid #444;">
                <a href="https://arxiv.org/pdf/2502.17239v1">Baichuan-Audio: A Unified Framework for End-to-End Speech Interaction</a>
            </th>
        </tr>
    </thead>
    <tbody>
        <tr style="background-color: white;">
            <td style="padding: 10px; border: 1px solid #ddd; text-align: center;">
                <img src="https://miro.medium.com/v2/resize:fit:1400/format:webp/1*Q-Wr1LsEOnt2JL6rxmpTAA.png" style="width: 100%; object-fit: contain;">
            </td>
            <td style="padding: 10px; border: 1px solid #ddd; text-align: center;">
                <img src="https://uploads-ssl.webflow.com/6696d42284cfe85e5e20165b/6696d86b9589667cc7ccb66c_63f85da8c5921e1f19fa69a5_Untitled.png" style="width: 100%; object-fit: contain;">
            </td>
            <td style="padding: 10px; border: 1px solid #ddd; text-align: center;">
                <img src="https://arxiv.org/html/2502.17239v1/x2.png" style="width: 100%;">
            </td>
        </tr>
        <tr style="vertical-align: top;">
            <td style="padding: 12px; border: 1px solid #ddd; background-color: #f9f9f9;">
                <b>Neural Audio Codec</b>: классический автоэнкодер. Задача — максимально компактно упаковать физику сигнала для передачи по сети и точного восстановления волны.
            </td>
            <td style="padding: 12px; border: 1px solid #ddd; background-color: #f9f9f9;">
                <b>Hierarchical Modeling</b>: вводит разделение на <b>семантические токены</b> и <b>акустические токены</b>.
            </td>
            <td style="padding: 12px; border: 1px solid #ddd; background-color: #f9f9f9;">
                <b>Aligned Multimodal LLM</b>: токенизатор интегрирован в LLM. Модель обучается понимать и генерировать речь как единый поток в рамках одного контекстного окна.
            </td>
        </tr>
        <tr style="vertical-align: top;">
            <td style="padding: 12px; border: 1px solid #ddd;">
                Использует <b>RVQ</b> — многослойное квантование. Обучается через <i>Reconstruction Loss</i> и <i>GAN</i>. Токены хранят акустику, но почти не содержат явной семантики.
            </td>
            <td style="padding: 12px; border: 1px solid #ddd;">
                Дискретизация скрытых слоев SSL-моделей (<b>w2v-BERT</b>, <b>HuBERT</b>) через <b>K-means</b>. Это дает высокоуровневые токены, которые стабильнее для длинной генерации, чем чистая акустика.
            </td>
            <td style="padding: 12px; border: 1px solid #ddd;">
                Использует <b>Finite Scalar Quantization</b>. Обучается через <b>Supervised Multi-task</b> (ASR + LID + SER). Токены сразу несут в себе и текст, и эмоциональный заряд.
            </td>
        </tr>
    </tbody>
</table>

## `2.2. Residual Vector Quantization (RVQ)`

Так как сжатие непрерывных векторов в малое дискретное пространство непременно приводит к потере информации, то возникает желание управлять trade-off между памятью и качеством. Один из способов усилить выразительную способность дискретного представление — использовать иерархическое кодирование:

**RVQ** приближает непрерывный вектор $\mathbf{h}$ суммой $N_{q}$ дискретных векторов из $N_{q}$ ($\approx 1-32$) независимых словарей размера $\approx 2^{10}$:
   * Первый квантователь $C_1$ выбирает токен, вектор которого ближе всего к $\mathbf{h}$: $\mathbf{c}_{1} = \arg\min\limits_{\mathbf{c}_{i} \in C_{1}}||\mathbf{h} - \mathbf{c}_i||_{2}$. Этот токен кодирует основную **семантическую** часть звука.
   * Второй квантователь $C_2$ кодирует остаток первого квантователя $\mathbf{r}_1 = \mathbf{h} - \mathbf{с}_{1}$, захватывая **тембр и интонацию**.
   * Третий ($C_3$) и последующие уровни кодируют мелкие акустические детали (фоновый шум, реверберацию).

<b style='color:red;'>В чем заключается проблема использования RVQ-токенов в авторегрессионных LLM?</b>

<details><summary>Ответ:</summary>>> 
В классических текстовых LLM текст — это одномерная последовательность: 1 токен на 1 временной шаг. А в кодеке на один шаг времени приходится набор из $N_{q}$ токенов. 

Если предсказывать их авторегрессионно один за другим, развернув в длинную цепочку, последовательность станет в $N_{q}$ раз длиннее, и вычисление Attention станет невозможным из-за квадратичной сложности. Если предсказывать параллельно — нарушаются причинно-следственные связи внутри кадра.

Для решения этой проблемы можно отказаться от точной авторегрессии в пользу [Inexact autoregressive decomposition](https://arxiv.org/pdf/2306.05284):

<img src="https://miro.medium.com/v2/resize:fit:1400/format:webp/1*uBeLtV1rY-IFRuXCnWHSMg.png">

</details>

# `3. Акустические модели`

Теперь, рассмотрим как можно решать задачу перевода текста в аудио представление. Акустическая модель решает задачу маппинга: как перевести лингвистическое представление в акустическое.

## `3.1. Авторегрессионные модели (AR) + Mel-спектрограммы`

Классический подход — использовать Sequence-to-Sequence архитектуру для предсказания непрерывных Mel-спектрограмм фрейм за фреймом:

1. **Encoder** кодирует текст.
2. **Attention** пытается понять, какое слово сейчас произносится, устанавливая *soft alignment* между текстом и генерируемым звуком.
3. **Decoder** авторегрессионно генерирует спектрограмму.

Главный минус: **Нестабильность и медленный инференс**. 

<b style='color:red;'>Почему AR-модели на базе механизма Attention часто пропускают слова или начинают бесконечно повторять одну и ту же фонему?</b>

<details><summary>Ответ:</summary>>> 
    
Во время инференса модель должна понять, когда пора перевести фокус внимания на следующую букву. Если в тексте встречаются сложные, длинные или необычные слова, фокус либо застрянет на одной букве, либо перескочит через несколько слов. Из-за авторегрессионной природы ошибка на одном шаге разрушает дальнейшую генерацию.
</details>

Можно выделить три особенности непрерывных авторегрессионных моделей:
1. Аналогично LLM, TTS модель должна предсказывать <eos> токен для остановки генерации. Для этого все AR модели на непрерывных представлениях содержат отдельную Stop Token голову, предсказывающую вероятность остановки.
   
<b style='color:red;'>Как должна выглядеть функция потерь для Stop Token головы?</b>

<details><summary>Ответ:</summary>>> 
    
Можно использовать обычный Cross-entropy loss, однако, нужно учитывать значительный дисбаланс между MSE/L1 лоссом для спектрограммы и Stop loss. Последний обычно нужно усилить. 
</details>

2. Большинство AR на непрерывных представлениях используют особую *свёрточную* Post-net, чтобы улучшить и **сгладить полностью предсказанную AR моделью спектрограмму**.

3. Так как непрерывная спектрограмма имеет значительную корреляцию между соседними фреймами, модели может быть выгодно предсказывать спектр текущего фрейма на следующем шаге, что приводит к очевидно некорректной генерации. Чтобы исключить застревание модели в таком локальном оптимуме нужно принудительно использовать глобальный контекст. В частности, можно использовать Dropout как на обучении, так и на инференсе. Для более старых моделей Dropout $\approx 0.5$ показывал наилучшие результаты. В более новых моделях Dropout частично заменяется на стохастический forward pass, например, через Latent Sampling Module. 

<table style="width: 100%; border-collapse: collapse; font-family: sans-serif; table-layout: fixed; text-align: left;">
    <thead>
        <tr style="background-color: #f2f2f2; border-bottom: 2px solid #444;">
            <th style="padding: 15px; width: 33.33%; border: 1px solid #ddd; text-align: center;">
                <a href="https://arxiv.org/pdf/1712.05884" style="text-decoration: none; color: #0066cc;">Natural TTS Synthesis by Conditioning WaveNet on Mel Spectrogram Predictions (Tacotron 2)</a>
            </th>
            <th style="padding: 15px; width: 33.33%; border: 1px solid #ddd; text-align: center;">
                <a href="https://arxiv.org/pdf/1809.08895" style="text-decoration: none; color: #0066cc;">Neural Speech Synthesis with Transformer Network (Transformer TTS)</a>
            </th>
            <th style="padding: 15px; width: 33.33%; border: 1px solid #ddd; text-align: center;">
                <a href="https://arxiv.org/pdf/2407.08551" style="text-decoration: none; color: #0066cc;">Autoregressive Speech Synthesis without Vector Quantization (MELLE)</a>
            </th>
        </tr>
    </thead>
    <tbody>
        <tr style="background-color: white;">
            <td style="padding: 10px; border: 1px solid #ddd; text-align: center; vertical-align: middle;">
                <img src="https://ar5iv.labs.arxiv.org/html/1809.08895/assets/Tacotron2.jpg" alt="Tacotron 2" style="max-width: 100%; max-height: 250px; object-fit: contain;">
            </td>
            <td style="padding: 10px; border: 1px solid #ddd; text-align: center; vertical-align: middle;">
                <img src="https://ar5iv.labs.arxiv.org/html/1809.08895/assets/Transformer.jpg" alt="Transformer TTS" style="max-width: 100%; max-height: 250px; object-fit: contain;">
            </td>
            <td style="padding: 10px; border: 1px solid #ddd; text-align: center; vertical-align: middle;">
                <img src="https://arxiv.org/html/2407.08551v2/x1.png" alt="MELLE" style="max-width: 100%; max-height: 250px; object-fit: contain;">
            </td>
        </tr>
        <tr style="vertical-align: top; background-color: #f9f9f9;">
            <td style="padding: 12px; border: 1px solid #ddd;">
                Seq2Seq модель на базе <b>LSTM</b>. Использует <i>Location-Sensitive Attention</i> для выравнивания текста и звука.
            </td>
            <td style="padding: 12px; border: 1px solid #ddd;">
                Seq2Seq модель, полностью построенная на <b>Multi-Head Self-Attention</b>, повторяющая оригинальную архитектуру Encoder-Decoder Transformer.
            </td>
            <td style="padding: 12px; border: 1px solid #ddd;">
                <b>Decoder-only LLM</b>, но работающая напрямую с <b>непрерывными</b> значениями мел-спектрограмм.
            </td>
        </tr>
        <tr style="vertical-align: top;">
            <td style="padding: 12px; border: 1px solid #ddd;">
                Энкодер переводит текст в скрытые представления. На каждом шаге декодер смотрит на предыдущий сгенерированный кадр спектрограммы, через Attention выбирает нужную фонему и авторегрессионно предсказывает следующий кадр.
            </td>
            <td style="padding: 12px; border: 1px solid #ddd;">
                Декодер через Cross-Attention использует скрытые представления текста из энкодера и авторегрессионно выдаёт кадры спектрограммы. Attention позволяет связывать отдаленные контексты.
            </td>
            <td style="padding: 12px; border: 1px solid #ddd;">
                Модель решает задачу In-Context Learning (Voice Cloning). MELLE генерирует параметры распределения ($\mathcal{N}(\mu, \sigma)$), из которого сэмплируется латентный вектор, превращающийся в <b>фреймы спектрограммы</b>. За один шаг может предсказывать сразу несколько (<i>r</i>) кадров.
            </td>
        </tr>
        <tr style="vertical-align: top; background-color: #f9f9f9;">
            <td style="padding: 12px; border: 1px solid #ddd;">
                <ul>
                    <li>Обучается через <i>Teacher-Forcing</i>.</li>
                    <li>Оптимизируется L1/MSE loss спектрограмм.</li>
                    <li>Обучение очень медленное из-за рекуррентных LSTM-слоев.</li>
                </ul>
            </td>
            <td style="padding: 12px; border: 1px solid #ddd;">
                <ul>
                    <li>Благодаря Self-Attention градиенты на этапе Teacher-Forcing считаются параллельно.</li>
                    <li>Требует специального <i>Scaled Positional Encoding</i> для выравнивания пространств текста и звука.</li>
                </ul>
            </td>
            <td style="padding: 12px; border: 1px solid #ddd;">
                <ul>
                    <li>Оптимизируется <b>Regression Loss</b> (L1 + MSE).</li>
                    <li><b>Spectrogram Flux Loss</b> поощряет генерацию разнообразной спектрограммы, защищая от сходимости в локальный оптимум.</li>
                </ul>
            </td>
        </tr>
        <tr style="vertical-align: top;">
            <td style="padding: 12px; border: 1px solid #ddd;">
                <b>+</b> Высокая естественность.<br>
                <b>-</b> При инференсе часто ломается Attention (проглатывания/повторения).<br>
                <b>-</b> Очень медленная авторегрессионная генерация.
            </td>
            <td style="padding: 12px; border: 1px solid #ddd;">
                <b>+</b> Интонация строится глобальнее благодаря Self-Attention.<br>
                <b>-</b> При инференсе часто ломается Attention (проглатывания/повторения).<br>
                <b>-</b> Очень медленная авторегрессионная генерация.
            </td>
            <td style="padding: 12px; border: 1px solid #ddd;">
                <b>+</b> Можно ускорять инференс, предсказывая по <i>r</i> кадров за раз.<br>
            </td>
        </tr>
    </tbody>
</table>

## `3.2. Неавторегрессионные модели (NAR) + Mel-спектрограммы`

Ключевой проблемой для AR моделей является медленный инференс, а также общая нестабильность связанная с накоплением ошибок и ошибками в **Soft Alignment** механизме внимания на Out-of-Domain данных. Переход к **Hard Alignment** позволяет решить эти проблемы. Общую архитектуру можно описать следующим образом:

1. **Encoder** кодирует текст.
2. **Length Regulator** предсказывает длительность каждой фонемы и дублирует скрытые представления фонем пропорционально их предсказанной длительности.
3. **NAR Decoder** параллельно генерирует всю Mel-спектрограмму за один проход.

Главный плюс: **Сверхбыстрый инференс**. 

<table style="width: 100%; border-collapse: collapse; font-family: sans-serif; table-layout: fixed; text-align: left;">
    <thead>
        <tr style="background-color: #f2f2f2; border-bottom: 2px solid #444;">
            <th style="padding: 15px; width: 33.33%; border: 1px solid #ddd; text-align: center;">
                <a href="https://arxiv.org/pdf/1905.09263" style="text-decoration: none; color: #0066cc;">FastSpeech: Fast, Robust and Controllable Text to Speech</a>
            </th>
            <th style="padding: 15px; width: 33.33%; border: 1px solid #ddd; text-align: center;">
                <a href="https://arxiv.org/pdf/2105.06337" style="text-decoration: none; color: #0066cc;">Grad-TTS: A Diffusion Probabilistic Model for TTS</a>
            </th>
            <th style="padding: 15px; width: 33.33%; border: 1px solid #ddd; text-align: center;">
                <a href="https://arxiv.org/pdf/2309.03199" style="text-decoration: none; color: #0066cc;">Matcha-TTS: A Fast TTS Architecture with Conditional Flow Matching</a>
            </th>
        </tr>
    </thead>
    <tbody>
        <tr style="background-color: white;">
            <td style="padding: 10px; border: 1px solid #ddd; text-align: center; vertical-align: middle;">
                <img src="https://publish-01.obsidian.md/access/93d590b5e88c06a4619cd08dc2123a4d/slp/9%20Speech%20Synthesis/attachments/ren-fastspeech1-arch.png" alt="FastSpeech" style="max-width: 100%; max-height: 250px; object-fit: contain;">
            </td>
            <td style="padding: 10px; border: 1px solid #ddd; text-align: center; vertical-align: middle;">
                <img src="https://ar5iv.labs.arxiv.org/html/2105.06337/assets/grad-tts-new.png" alt="Grad-TTS" style="max-width: 100%; max-height: 250px; object-fit: contain;">
            </td>
            <td style="padding: 10px; border: 1px solid #ddd; text-align: center; vertical-align: middle;">
                <img src="https://cdn-ak.f.st-hatena.com/images/fotolife/N/Ninhydrin/20230927/20230927085934.png" alt="Matcha-TTS" style="max-width: 100%; max-height: 250px; object-fit: contain;">
            </td>
        </tr>
        <tr style="vertical-align: top; background-color: #f9f9f9;">
            <td style="padding: 12px; border: 1px solid #ddd;">
                <b>NAR</b> модель на базе Transformer. Обучается напрямую предсказывать Mel-спектрограмму.
            </td>
            <td style="padding: 12px; border: 1px solid #ddd;">
                <b>Score-based Diffusion</b>. Акустическая модель предсказывает параметры распределения, а диффузионная сеть итеративно превращает зашумлённое представление звука в спектрограмму.
            </td>
            <td style="padding: 12px; border: 1px solid #ddd;">
                <b>Conditional Flow Matching</b>. Самая современная замена классической диффузии на базе Оптимального Транспорта.
            </td>
        </tr>
        <tr style="vertical-align: top;">
            <td style="padding: 12px; border: 1px solid #ddd;">
                Фонемы подаются в <i>Variance Adaptor</i>, который предсказывает длительность. Затем вектор растягивается и декодер параллельно выдаёт все кадры спектрограммы.
            </td>
            <td style="padding: 12px; border: 1px solid #ddd;">
                Энкодер переводит текст в $\mu$ — среднее значение для нормального распределения. Спектрограмма генерируется путём постепенного удаления шума из этого распределения в сторону реальных акустических признаков.
            </td>
            <td style="padding: 12px; border: 1px solid #ddd;">
                Модель не предсказывает шум, как диффузия, а выучивает <i>векторное поле</i> — прямые траектории, по которым шум переходит в спектрограмму. Текст и длительности фонем выступают в качестве условия.
            </td>
        </tr>
        <tr style="vertical-align: top; background-color: #f9f9f9;">
            <td style="padding: 12px; border: 1px solid #ddd;">
                <ul>
                    <li>Сверхбыстрое параллельное обучение.</li>
                    <li>Оптимизируются L1/MSE функции потерь для спектрограммы и длительности.</li>
                </ul>
            </td>
            <td style="padding: 12px; border: 1px solid #ddd;">
                <ul>
                    <li>Отказ от жесткого MSE для спектрограммы в пользу стохастического диффузионного лосса.</li>
                    <li>Требует решения SDE на этапе инференса.</li>
                </ul>
            </td>
            <td style="padding: 12px; border: 1px solid #ddd;">
                <ul>
                    <li>Обучение сводится к регрессии векторного поля.</li>
                </ul>
            </td>
        </tr>
        <tr style="vertical-align: top;">
            <td style="padding: 12px; border: 1px solid #ddd;">
                <b>+</b> Высокая стабильность (нет пропусков слов).<br>
                <b>+</b> Сверхбыстрый инференс.<br>
                <b>-</b> Проблема <i>oversmoothing</i>: усреднение лосса делает голос слегка "роботизированным" и глухим.
            </td>
            <td style="padding: 12px; border: 1px solid #ddd;">
                <b>+</b> Решает проблему oversmoothing.<br>
                <b>-</b> Медленная генерация (требует 50–100 проходов нейросети для денойзинга).
            </td>
            <td style="padding: 12px; border: 1px solid #ddd;">
                <b>+</b> Качество неотличимо от диффузии (очень высокая естественность).<br>
                <b>+</b> Сверхбыстрый инференс: достаточно всего <b>2-4 шагов</b> ODE-солвера.<br>
                <b>+</b> Очень "лёгкая" модель по памяти.
            </td>
        </tr>
    </tbody>
</table>

### `Обучение Length Regulator и проблема выравнивания`

Основной вопрос в NAR моделях — это как обучать **Length Regulator**. Чтобы растянуть фонемные представления, нам нужно знать точную длительность каждой фонемы в кадрах спектрограммы. Но в сырых датасетах обычно есть только текст и аудио.

<b style='color:red;'>Почему нельзя использовать Soft Attention внутри NAR-модели, чтобы она сама "размазала" фокус внимания по спектрограмме без явного Length Regulator?</b>

<details><summary>Ответ:</summary>>> 
    
1. **Потеря темпоральной структуры:** Soft Attention присваивает вероятности. Фрейм спектрограммы может на 40% состоять из звука «А» и на 60% из звука «Б». Это приведет к генерации смеси звуков. Для правильного дублирования токенов во времени нужно предсказывать точные границы.
2. **Нарушение монотонности:** Обычный Attention ничем не ограничен. Он может сначала посмотреть на конец предложения, а потом на начало. В речи фонемы всегда идут строго одна за другой. 
</details>

* **Дистилляция из AR-учителя:**
   
Так Attention-based AR модели самостоятельно выучивают Soft Alignment его можно использовать для генерации Hard Alignment. В модели FastSpeech 1 было предложено обучать AR модель. Из её матрицы внимания извлекается наиболее вероятный путь, который и используется как Ground Truth длительности для обучения NAR-модели.
* **Внешние инструменты (Forced Alignment):**

Очевидно, что обучение AR модели требует значительных лишних ресурсов. Поэтому можно использовать классические методы для определения границ фонем. Так, например, в FastSpeech 2 использовался **MFA (Montreal Forced Aligner)**. Это классическая система на базе скрытых марковских моделей, которая с высокой точностью находит границы фонем.
   * *Плюс:* Очень стабильно, работает из коробки.
   * *Минус:* Усложняет пайплайн. MFA — отдельная система, требующая своего обучения, словарей произношений и настройки.

* **End-to-End выравнивание (MAS):**
  
Начиная с моделей Glow-TTS, VITS и современных диффузионных сетей (Matcha-TTS), индустрия перешла к обучению выравнивания без внешних инструментов напрямую в процессе тренировки модели. Для этого используется алгоритм **Monotonic Alignment Search (MAS)**.
   Модель вычисляет матрицу вероятностей того, что каждый аудио-фрейм соответствует каждой фонеме. MAS использует динамическое программирование, чтобы найти наиболее вероятный *монотонный* путь в этой матрице. Найденный путь используется как "жесткая" разметка длительностей на текущем шаге обучения.

<b style='color:red;'>В чем основная проблема MAS на ранних этапах обучения?</b>
<details><summary>Ответ:</summary>>>
<b>"Cold Start" проблема.</b> В самом начале обучения веса модели случайны, и матрица правдоподобия представляет собой шум. MAS может найти случайный путь, и модель начнет учиться на неправильном выравнивании. Чтобы этого избежать, сначала обучают модель на коротких фразах, где ошибиться сложнее.
</details>

![](https://open-speech-ekstep.github.io/img/glow_training_and_inference.png)

### `Controllability и Variance Adaptors`

В NAR моделях базовым механизмом управления характеристиками речи выступают Variance Adaptors, предложенные в архитектуре FastSpeech 2. Данные модули состоят из предикторов, прогнозирующих длительность (Duration), фундаментальную частоту (Pitch / F0) и энергию (Energy) для каждой фонемы на основе скрытых представлений текста. На этапе инференса скалярное умножение выходов этих предикторов на заданные коэффициенты позволяет контролировать скорость и высоту тона генерируемой речи.

Мы обсуждали, что отсутствие характеристик речи на обучении приводит к эффекту oversmoothing: Pitch и Energy, неявно выучиваемая моделью сходятся к средним значением, из-за чего синтезированная речь звучит монотонно и неестественно.

Эту проблему можно решить, добавив блоки, которые помимо длины фонемы предсказывают её Pitch, Energy и другие свойства. Это позволяет распутать произношение разных фонем, а также добавить простое пофонемное управление генерацией. 

<b style='color:red;'>Откуда взять обучающие данные для блоков-предикторов?</b>

<details><summary>Ответ:</summary>>> 

Существуют как классические, так и нейросетевые методы для предсказания F0. Данные предсказания и выступают явным таргетом на обучении. Для энергии достаточно взять норму соответствующего фрейма после STFT.
    
</details>

![](https://www.chengfeng.wiki/images/fs2.png)

Для управления общим стилем речи можно добавить произвольные эмбеддинги, определяющие конкретного спикера в процессе обучения. Например, в модели [DiffStyleTTS](https://arxiv.org/pdf/2412.03388) предлагается иерархическое разделение управляющих факторов на три уровня:
1.  **Speaker Identity (Timbre):** Идентичность спикера. В данной модели реализована через таблицу эмбеддингов фиксированного размера для спикеров из обучающей выборки.
2.  **Implicit Style (GST):** Глобальные стилистические характеристики (эмоциональный окрас, акустические условия). Извлекаются из референсного аудиосигнала обучаемой сетью.
3.  **Explicit Prosody (Pitch, Energy, Duration):** Явные физические параметры. Предсказываются явно моделью, обусловленной на текст и стилевые вектора.

<b style='color:red;'>Почему в архитектуре DiffStyleTTS функция потерь вычисляется для Pitch и Energy, но не применяется напрямую к Speaker Embedding или Implicit Style Condition?</b>

<details><summary>Ответ:</summary>>> 
    
Pitch, Energy и Duration выступают целевыми переменными, которые акустическая модель должна уметь синтезировать на этапе инференса, имея на входе только текст. Speaker ID и неявный стиль являются входными условиями, задаваемыми извне (через референсное аудио или идентификатор). Оптимизация самих условий напрямую не требуется; их корректность и репрезентативность оцениваются опосредованно через функцию потерь восстановления финальной мел-спектрограммы.
</details>

<b style='color:red;'>Обладает ли данная архитектура способностью к Zero-Shot синтезу: генерации речи с характеристиками спикера, отсутствовавшего в обучающей выборке?</b>

<details><summary>Ответ:</summary>>> 
    
Разрешение этого вопроса требует разделения характеристик:
1. <b>Zero-Shot перенос просодии (Style Transfer) — возможен.</b> Модуль GST способен извлекать неявный стиль из произвольного OOD (Out-of-Domain) референсного аудио и переносить интонационный контур на целевой текст.
2. <b>Zero-Shot клонирование тембра (Voice Cloning) — невозможно.</b> Моделирование идентичности спикера через <i>Lookup Table</i> строго ограничивает генерацию тембрами из обучающей выборки. 

Для реализации полноценного Zero-Shot клонирования необходимо заменить Lookup Table на предварительно обученный дискриминативный энкодер спикера (например, на базе архитектур WavLM, ECAPA-TDNN), способный извлекать инвариантный вектор тембра из произвольного аудиосигнала.
</details>

![](https://xuan3986.github.io/DiffStyleTTS/demos/DiffStyleTTS.png)

## `3.3. Авторегрессионные модели (AR) + Дискретные токены`

Появление нейронных аудиокодеков (EnCodec, SoundStream) позволило сжимать непрерывную волну в последовательность дискретных токенов: **TTS превратился в задачу языкового моделирования**. Теперь можно подать в модель текстовый промпт, аудио-референс в виде токенов и целевой текст. Модель авторегрессионно продолжит последовательность токенов, опираясь на контекст, тем самым копируя голос, эмоцию и даже акустику помещения (Zero-Shot In-Context Learning).

Вместо обучения на чистых студийных данных, эти модели обучаются на десятках тысяч часов грязных данных (подкасты, аудиокниги, YouTube). Обучение сводится к предсказанию следующего токена с учетом уже сгенерированных (или поданных на вход) токенов.

Главный плюс: **Zero-Shot Voice Cloning**. 
Модели подаётся текстовый промпт, короткий фрагмент голоса целевого спикера (обычно ~3 секунды) в виде аудиотокенов и целевой текст. Модель авторегрессионно продолжает последовательность аудиотокенов, копируя тембр, акустику помещения и манеру речи из промпта без дообучения под нового спикера.

<table style="width: 100%; border-collapse: collapse; font-family: sans-serif; table-layout: fixed; text-align: left;">
    <thead>
        <tr style="background-color: #f2f2f2; border-bottom: 2px solid #444;">
            <th style="padding: 15px; width: 33.33%; border: 1px solid #ddd; text-align: center;">
                <a href="https://arxiv.org/pdf/2301.02111" style="text-decoration: none; color: #0066cc;">Neural Codec Language Models are Zero-Shot TTS Synthesizers (VALL-E)</a>
            </th>
            <th style="padding: 15px; width: 33.33%; border: 1px solid #ddd; text-align: center;">
                <a href="https://arxiv.org/pdf/2403.16973" style="text-decoration: none; color: #0066cc;">VoiceCraft: Zero-Shot Speech Editing and TTS in the Wild</a>
            </th>
            <th style="padding: 15px; width: 33.33%; border: 1px solid #ddd; text-align: center;">
                <a href="https://arxiv.org/pdf/2402.08093" style="text-decoration: none; color: #0066cc;">BASE TTS: Lessons from building a billion-parameter TTS model</a>
            </th>
        </tr>
    </thead>
    <tbody>
        <tr style="background-color: white;">
            <td style="padding: 10px; border: 1px solid #ddd; text-align: center; vertical-align: middle;">
                <img src="https://www.zauberware.com/assets/header/Screenshot-2023-01-11-at-22.29.37.png" alt="VALL-E" style="max-width: 100%; max-height: 250px; object-fit: contain;">
            </td>
            <td style="padding: 10px; border: 1px solid #ddd; text-align: center; vertical-align: middle;">
                <img src="https://miro.medium.com/v2/resize:fit:1400/format:webp/0*7cUA7yB6WSif34gi" alt="VoiceCraft" style="max-width: 100%; max-height: 250px; object-fit: contain;">
            </td>
            <td style="padding: 10px; border: 1px solid #ddd; text-align: center; vertical-align: middle;">
                <img src="https://scx1.b-cdn.net/csz/news/800a/2024/amazon-unveils-largest.jpg" alt="BASE TTS" style="max-width: 100%; max-height: 250px; object-fit: contain;">
            </td>
        </tr>
        <tr style="vertical-align: top; background-color: #f9f9f9;">
            <td style="padding: 12px; border: 1px solid #ddd;">
                Первая TTS LLM модель. Состоит из двух стадий: <b>AR</b> (для 1-го слоя RVQ) + <b>NAR</b> (для слоев 2-8).
            </td>
            <td style="padding: 12px; border: 1px solid #ddd;">
                Единая Transformer Decoder <b>AR</b> модель с использованием механизма <i>Delayed Stacking</i> и <i>Causal Masking</i>.
            </td>
            <td style="padding: 12px; border: 1px solid #ddd;">
                <b>AR</b> модель с 1 миллиардом параметров. Обучалась на 100К часах данных. Отказ от RVQ в пользу однопоточных BPE-токенов.
            </td>
        </tr>
        <tr style="vertical-align: top;">
            <td style="padding: 12px; border: 1px solid #ddd;">
                Обучение сводится к языковому моделированию. Модель учится связывать BPE-токены текста и дискретные токены из EnCodec. Отлично копирует акустику промпта.
            </td>
            <td style="padding: 12px; border: 1px solid #ddd;">
                Позволяет не только продолжать речь (TTS), но и <b>редактировать</b> её. Модель маскирует слово внутри фразы и генерирует замену с учетом как левого, так и правого аудио-контекста.
            </td>
            <td style="padding: 12px; border: 1px solid #ddd;">
                Вместо сырых аудио-токенов использует скрытые состояния WavLM, прогнанные через векторное квантование и Byte-Pair Encoding (BPE). Это радикально сокращает длину последовательности.
            </td>
        </tr>
        <tr style="vertical-align: top; background-color: #f9f9f9;">
            <td style="padding: 12px; border: 1px solid #ddd;">
                <b>+</b> Одна из первых моделей с Zero-Shot клонированием из коробки.<br>
                <b>-</b> Присутствуют галлюцинации (ошибки в произношении) из-за слабого выравнивания текста и звука в AR.
            </td>
            <td style="padding: 12px; border: 1px solid #ddd;">
                <b>+</b> Высокое качество на задаче редактирования речи.<br>
                <b>-</b> Инференс достаточно ресурсоёмкий и сложный из-за перестановки токенов.
            </td>
            <td style="padding: 12px; border: 1px solid #ddd;">
                <b>+</b> Демонстрация <i>эмерджентных свойств</i>: модель начинает правильно интонировать сложные синтаксические конструкции и передавать эмоции, опираясь только на текстовый контекст.<br>
                <b>-</b> Требует колоссальных ресурсов.
            </td>
        </tr>
    </tbody>
</table>

## `3.4. Неавторегрессионные модели (NAR) + Токены / Латентные представления`

Интеграция дискретных токенов и непрерывных латентных представлений с неавторегрессионной (NAR) генерацией направлена на решение проблемы вычислительной сложности AR-моделей при сохранении высокой естественности и Zero-Shot возможностей. Вместо последовательного предсказания следующего токена, данные архитектуры генерируют всю последовательность параллельно. Для этого используются методы итеративного уточнения: **Masked Audio Modeling**  или **Diffusion / Flow Matching**.

<b style='color:red;'>Зачем нужно промежуточное представление в виде семантических токенов, извлекаемых из ASR-моделей, отказываясь от прямого маппинга «Текст $\to$ Акустика»?</b>

<details><summary>Ответ:</summary>>> 
    
Причина заключается в проблеме **Information Gap**. Прямой маппинг из текста в акустическое представление (которое содержит не только произношение, но и тембр, фоновый шум, микроинтонации) является слишком сложной задачей для NAR-моделей, что часто ведет к галлюцинациям и потере выравнивания.
<br><br>
Семантические токены выступают строгим информационным мостом: они обладают жесткой привязкой к лингвистическому смыслу и базовой просодии, но инвариантны к тембру и акустическим шумам. Использование каскадного пайплайна <code>Text $\to$ Semantic $\to$ Acoustic</code> стабилизирует процесс генерации, так как каждая подсеть решает узкоспециализированную задачу.
</details>

<table style="width: 100%; border-collapse: collapse; font-family: sans-serif; table-layout: fixed; text-align: left;">
    <thead>
        <tr style="background-color: #f2f2f2; border-bottom: 2px solid #444;">
            <th style="padding: 15px; width: 33.33%; border: 1px solid #ddd; text-align: center;">
                <a href="https://arxiv.org/pdf/2305.09636" style="text-decoration: none; color: #0066cc;">SoundStorm: Efficient Parallel Audio Generation</a>
            </th>
            <th style="padding: 15px; width: 33.33%; border: 1px solid #ddd; text-align: center;">
                <a href="https://arxiv.org/pdf/2403.03100" style="text-decoration: none; color: #0066cc;">NaturalSpeech 3: Zero-Shot TTS with Factorized Codec and Diffusion</a>
            </th>
            <th style="padding: 15px; width: 33.33%; border: 1px solid #ddd; text-align: center;">
                <a href="https://arxiv.org/pdf/2407.08551" style="text-decoration: none; color: #0066cc;">CosyVoice: A Scalable Multilingual Zero-shot TTS Synthesizer</a>
            </th>
        </tr>
    </thead>
    <tbody>
        <tr style="background-color: white;">
            <td style="padding: 10px; border: 1px solid #ddd; text-align: center; vertical-align: middle;">
                <img src="https://blogger.googleusercontent.com/img/b/R29vZ2xl/AVvXsEjI75okNCRdLNSwU31uccP6wuIXVDxhhhvTI9f_V0ntiAxUOtw29TZeAvNFZXNBta9GraHB1bikM6zGQrFM7zSt2d985P6OC9On0xDwrLfj9OzJjdJ8BjrcYiAj_VGi3VQsqCIbNx_B7DP75KCU9D6xB_ybgHDpBk0z-eBkXXRdu9u04zcdzNCKGJuqszrR/s16000/image2.png" alt="SoundStorm" style="max-width: 100%; max-height: 250px; object-fit: contain;">
            </td>
            <td style="padding: 10px; border: 1px solid #ddd; text-align: center; vertical-align: middle;">
                <img src="https://miro.medium.com/v2/resize:fit:1400/format:webp/0*LqcJdrKlt1T8W9S3.jpg" alt="NaturalSpeech 3" style="max-width: 100%; max-height: 250px; object-fit: contain;">
            </td>
            <td style="padding: 10px; border: 1px solid #ddd; text-align: center; vertical-align: middle;">
                <img src="https://arxiv.org/html/2407.05407v2/x1.png" alt="CosyVoice" style="max-width: 100%; max-height: 250px; object-fit: contain;">
            </td>
        </tr>
        <tr style="vertical-align: top; background-color: #f9f9f9;">
            <td style="padding: 12px; border: 1px solid #ddd;">
                <b>NAR</b> архитектура на дискретных токенах. Использует концепцию <i>Masked Audio Modeling</i>.
            </td>
            <td style="padding: 12px; border: 1px solid #ddd;">
                <b>NAR</b> архитектура в непрерывном пространстве латентных представлений. Основана на <i>Latent Diffusion</i> и строгой <b>Факторизации</b> атрибутов.
            </td>
            <td style="padding: 12px; border: 1px solid #ddd;">
                <b>Гибридная архитектура</b>: AR-модель для генерации семантических токенов + NAR-модель (<i>Conditional Flow Matching</i>) для синтеза спектрограммы.
            </td>
        </tr>
        <tr style="vertical-align: top;">
            <td style="padding: 12px; border: 1px solid #ddd;">
                Опирается на семантические токены в качестве условия. Сама модель заменяет медленную AR-часть, генерируя слои RVQ-токенов параллельно. Начинает с полностью замаскированной последовательности и итеративно предсказывает токены.
            </td>
            <td style="padding: 12px; border: 1px solid #ddd;">
                Сложный речевой сигнал разбивается на независимые подпространства: <i>Content, Prosody, Timbre, Acoustic Details</i>. Для каждого атрибута обучается своя независимая диффузионная сеть.
            </td>
            <td style="padding: 12px; border: 1px solid #ddd;">
                Использует Supervised Semantic Tokens из ASR-модели SenseVoice. LLM предсказывает семантику на основе текста, а Flow Matching модель детерминированно переводит шум в спектрограмму, опираясь на семантические токены и эмбеддинг спикера.
            </td>
        </tr>
        <tr style="vertical-align: top; background-color: #f9f9f9;">
            <td style="padding: 12px; border: 1px solid #ddd;">
                <ul>
                    <li>Синтез аудио происходит на 2 порядка быстрее, чем в AR-кодековых моделях.</li>
                    <li>Полностью решает проблему зацикливаний и пропусков.</li>
                </ul>
            </td>
            <td style="padding: 12px; border: 1px solid #ddd;">
                <ul>
                    <li>Высокий уровень модульного контроля (возможность комбинировать тембр одного спикера с просодией другого).</li>
                </ul>
            </td>
            <td style="padding: 12px; border: 1px solid #ddd;">
                <ul>
                    <li>Flow-matching значительно быстрее классической диффузии.</li>
                </ul>
            </td>
        </tr>
        <tr style="vertical-align: top;">
            <td style="padding: 12px; border: 1px solid #ddd;">
                <b>+</b> Высочайшая производительность (генерация 30 секунд аудио за 0.5 сек).<br>
            </td>
            <td style="padding: 12px; border: 1px solid #ddd;">
                <b>+</b> Решение проблемы entangled признаков.<br>
                <b>-</b> Экстремально сложный пайплайн обучения и декомпозиции аудио.
            </td>
            <td style="padding: 12px; border: 1px solid #ddd;">
                <b>+</b> Высокое качество Zero-Shot Voice Cloning (включая кросс-языковой перенос).<br>
                <b>-</b> Наличие AR-модуля для семантики оставляет теоретический риск галлюцинаций.
            </td>
        </tr>
    </tbody>
</table>

## `3.5. Пример генерации`

In [ ]:
def get_speaker_embedding(audio_path):
    signal, fs = soundfile.read(audio_path)
    signal = torch.tensor(signal)
    signal = torchaudio.functional.resample(signal, orig_freq=fs, new_freq=16000)
    
    with torch.no_grad():
        embeddings = speaker_model.encode_batch(signal)
        embeddings = F.normalize(embeddings, dim=-1).squeeze(0)
    return embeddings.to(device)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

processor = SpeechT5Processor.from_pretrained("microsoft/speecht5_tts")
model = SpeechT5ForTextToSpeech.from_pretrained("microsoft/speecht5_tts").to(device)
vocoder = SpeechT5HifiGan.from_pretrained("microsoft/speecht5_hifigan").to(device)
speaker_model = EncoderClassifier.from_hparams(
    source="speechbrain/spkrec-xvect-voxceleb", 
    run_opts={"device": device}, 
    savedir="tmp_spkrec"
)

In [ ]:
reference_audio_path = "./audio_test.wav" 
text = "Hello, students! Today, we are learning about zero-shot voice cloning."

with torch.no_grad():
    inputs = processor(text=text, return_tensors="pt").to(device)
    speaker_embeddings = get_speaker_embedding(reference_audio_path)
    speech = model.generate_speech(inputs["input_ids"], speaker_embeddings, vocoder=vocoder)

Audio(speech.cpu().numpy(), rate=16000)